# Soup Kernel Challenge: speed candidate, <=2000 MiB target

Experimental code: GPU time and peak are not yet verified. Target: <65 s and <=2000 MiB.
Kaggle: GPU T4 x2, Internet on, fresh session. Only GPU 0 is used by the original harness.
The notebook runs two repeats per side and all original checks. No automatic background runs.
After completion, download the ZIP from the final cell.


In [ ]:
REPO_URL = "https://github.com/samenov01/soup-kernel-lab.git"
COMMIT = "c9a2eadb17d51efed112678061bf4e75ae14a048"  # v2: 192 MiB saved activations, reused gate/up backward operand


In [ ]:
import os, shutil, subprocess
if shutil.which("nvidia-smi") is None:
    raise RuntimeError("No NVIDIA runtime. Select GPU T4 x2 and start a fresh Kaggle session.")
gpus = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                      capture_output=True, text=True).stdout
print(gpus)
assert "T4" in gpus, "Switch the accelerator to GPU T4 x2"
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass
subprocess.run(["python", "-m", "pip", "install", "-q", "soup-cli[train]==0.77.0", "transformers==5.16.1", "peft==0.20.0", "trl==0.29.1", "bitsandbytes==0.50.2", "accelerate==1.14.0", "pytest"], check=True)


In [ ]:
import base64, os, shutil, subprocess
from pathlib import Path
if Path("/kaggle/working/repo").exists():
    raise RuntimeError("repo already exists. Preserve any edits and outputs, then use a fresh Kaggle session.")
token = ""
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception as exc:
    print("No GITHUB_TOKEN secret, cloning without it:", type(exc).__name__)
auth = []
if token:  # the token goes in a header for this one command: never in a URL, a file or the output
    basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    auth = ["-c", f"http.extraHeader=Authorization: Basic {basic}"]
clone = subprocess.run(["git", *auth, "clone", "-q", REPO_URL, "/kaggle/working/repo"],
                       capture_output=True, text=True)
if clone.returncode:
    raise SystemExit("git clone failed: " + clone.stderr)
if COMMIT:
    subprocess.run(["git", "-C", "/kaggle/working/repo", "checkout", "-q", COMMIT], check=True)
print(subprocess.run(["git", "-C", "/kaggle/working/repo", "log", "-1", "--format=%H %s"],
                     capture_output=True, text=True).stdout)

In [ ]:
from pathlib import Path
assert Path("/kaggle/working/repo/solution/host_offload.py").exists(), "Check REPO_URL and COMMIT"
subprocess.run(["python", "task/prepare_data.py"], cwd="/kaggle/working/repo", check=True)
subprocess.run(["python", "-m", "pytest", "solution", "-q"], cwd="/kaggle/working/repo", check=True)


In [ ]:
subprocess.run(["python", "harness/run_task.py", "--repeats", "2", "--out", "runs"], cwd="/kaggle/working/repo", check=True)


In [ ]:
subprocess.run(["python", "harness/check_rules.py", "--result", "runs/result.json"], cwd="/kaggle/working/repo", check=True)
subprocess.run(["python", "harness/score.py", "runs/result.json"], cwd="/kaggle/working/repo", check=True)


In [ ]:
%run /kaggle/working/repo/automation/archive_results.py
from IPython.display import FileLink, display
display(FileLink("/kaggle/working/kernel_results.zip"))
